In [1]:
import sys
sys.path.append(f"./../")
from quantum_walk_utils import *
from collections import defaultdict

In [2]:
class MatchingVsPauliAnalyzer:
   def __init__(self, n_qubits: int, delta_t: float, logger: Logger):
       self.analyzer = BaseAnalyzer(n_qubits, delta_t)
       self.logger = logger
       
   def analyze_graph(self, edges: Set[Tuple[str, str]], n_steps: int = 1) -> Optional[Dict]:
       self.logger.log("Computing Matchings Dynamic walk")
       matching_metrics = self.analyzer.analyze_matching(edges, n_steps)
       if not matching_metrics:
           self.logger.log("Failed to get matching metrics")
           return None
       self.logger.log_metrics("Matchings", 
                             matching_metrics.cx_count,
                             matching_metrics.u3_count,
                             matching_metrics.depth)
       
       self.logger.log("Computing Pauli decomposition")
       pauli_metrics = self.analyzer.analyze_pauli(edges)
       if not pauli_metrics:
           self.logger.log("Failed to get Pauli metrics")
           return None
       self.logger.log_metrics("Pauli",
                             pauli_metrics.cx_count,
                             pauli_metrics.u3_count,
                             pauli_metrics.depth)
       
       cx_diff = matching_metrics.cx_count - pauli_metrics.cx_count
       u3_diff = matching_metrics.u3_count - pauli_metrics.u3_count
       
       if cx_diff < 0:
           category = 'win'
       elif cx_diff > 0:
           category = 'lose'
       else:
           category = 'draw'
           
       return {
           'category': category,
           'matching_cx': matching_metrics.cx_count,
           'matching_u3': matching_metrics.u3_count,
           'pauli_cx': pauli_metrics.cx_count,
           'pauli_u3': pauli_metrics.u3_count,
           'cx_diff': cx_diff,
           'u3_diff': u3_diff,
           'matching_depth': matching_metrics.depth,
           'pauli_depth': pauli_metrics.depth
       }

In [3]:
# Configuration
delta_t = 0.1
dir = '../data/graphs/'

n_qubits = 3
n_vertices = 2**n_qubits

n_graphs = 11117
# n_graphs = 200
# type = 'bipartite'
# type = 'losing'
type = 'BM'
graph_file = dir+f'{n_graphs}graph_{type}_{n_vertices}c.g6'

In [4]:
# Setup
base_dir = '../outputs/matching_vs_pauli'
dirs = setup_directories(base_dir)
logger = Logger(os.path.join(dirs['logs'], f'analysis_{type}_{n_vertices}c.log'))

In [5]:
# Log initial parameters
logger.log(f"Starting analysis with:")
logger.log(f"Number of qubits: {n_qubits}")
logger.log(f"Number of vertices: {n_vertices}")
logger.log(f"Delta t: {delta_t}")
logger.log(f"Input file: {graph_file}")

In [6]:
try:
   if not os.path.exists(graph_file):
       raise FileNotFoundError(f"Graph file not found: {graph_file}")
   
   graph_info = GraphProcessor.parse_graph_filename(graph_file)
   logger.log(f"Parsed graph info: {graph_info}")
   
   analyzer = MatchingVsPauliAnalyzer(n_qubits, delta_t, logger)
   results_manager = ResultsManager(dirs['results'], graph_info)
   plot_manager = PlotManager(dirs['plots'])

   results = []
   categories = {'win': 0, 'lose': 0, 'draw': 0}
   original_graphs = []

   # Count total graphs
   with open(graph_file, 'r') as f:
       total_lines = sum(1 for _ in f)
   logger.log(f"Found {total_lines} graphs in file")

   # Process graphs
   with open(graph_file, 'r') as f:
       for i, line in enumerate(f):
           logger.log(f"\nProcessing graph {i}/{total_lines}")
           try:
               line = line.strip()
               if not line:
                   logger.log(f"Empty line at {i}, skipping")
                   continue
                   
               graph = nx.from_graph6_bytes(line.encode())
               logger.log(f"Graph {i} has {len(graph.nodes())} nodes and {len(graph.edges())} edges")
               
               edges = GraphProcessor.graph_to_bitstring(graph)
               logger.log(f"Converted to {len(edges)} bitstring edges")
               
               result = analyzer.analyze_graph(edges)
               if result:
                   categories[result['category']] += 1
                   results.append({'index': i, **result})
                   original_graphs.append(graph)
                   logger.log(f"Successfully processed graph {i} - Category: {result['category']}")
               else:
                   logger.log(f"Failed to analyze graph {i}")
                   
           except Exception as e:
               logger.log(f"Error processing graph {i}: {str(e)}")
               continue

   # Log final statistics
   logger.log("\nFinal Results Summary:")
   total_processed = sum(categories.values())
   logger.log(f"Total graphs in file: {total_lines}")
   logger.log(f"Successfully processed: {total_processed}")
   logger.log_final_stats(categories, total_processed)

   if total_processed > 0:
       # Separate results by category
       win_results = [r for r in results if r['category'] == 'win']
       lose_results = [r for r in results if r['category'] == 'lose']
       draw_results = [r for r in results if r['category'] == 'draw']

       # Calculate category averages
       def calc_category_avgs(cat_results, prefix):
           if not cat_results:
               return {}
           metrics = ['matching_cx', 'matching_u3', 'pauli_cx', 'pauli_u3', 
                     'matching_depth', 'pauli_depth']
           totals = defaultdict(float)
           for r in cat_results:
               for metric in metrics:
                   totals[metric] += r[metric]
           return {f"{prefix}_{k}": v/len(cat_results) for k, v in totals.items()}

       avg_stats = {}
       avg_stats.update(calc_category_avgs(win_results, 'win'))
       avg_stats.update(calc_category_avgs(lose_results, 'lose'))
       avg_stats.update(calc_category_avgs(draw_results, 'draw'))

       # Log averages
       logger.log("\nAverage Gate Counts by Category:")
       for metric, value in avg_stats.items():
           logger.log(f"{metric}: {value:.2f}")
       
       # Save results
       results_manager.save_results(categories, 'summary')
       results_manager.save_results({'detailed_results': results}, 'detailed')
       results_manager.save_results(avg_stats, 'averages')

       # Save categorized graphs
       results_manager.save_categorized_graphs(results, original_graphs)

       # Create visualization
       pie_chart = plot_manager.create_pie_chart(
           categories,
           f'Matching vs Pauli Results ({n_qubits} qubits)'
       )
       plot_manager.save_plot(pie_chart, f'pie_chart_{graph_info["size"]}_{graph_info["vertices"]}c.png')
   else:
       logger.log("No results to save - all graphs failed processing")

except Exception as e:
   logger.log(f"Critical error in main execution: {str(e)}")
   raise